In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import sys
import os
sys.path.append("../../")
sys.path.append("../../backend/src")

from rdflib.plugins.stores.sparqlstore import SPARQLStore

from backend.src.ontology import OntologyManager, OntologyConfig, Graph
from backend.src.datasetmatcher import DatasetManager
from backend.src.explorative.explorative_support import GuidanceManager
from backend.src.explorative.llm_query import (
    EnrichedEntitiesRelations,
    LLMQuery,
    QueryProgress,
)
from backend.src.eval_config import MultimodalConfig
from backend.src.explorative.topic_init import TopicInitator
from tqdm import tqdm
import pandas as pd

In [3]:
from backend.src.eval_config import get_config

# db_setups = [get_config("dbpedia", 1), get_config("oma", 1)]
db_setups = [ get_config("gutbrainie"), get_config("bto")]

In [11]:


for setup in db_setups:
    store = SPARQLStore(
        setup.sparql_endpoint,
        method="POST_FORM",
        params={"infer": False, "sameAs": False},
        retries=10,
    )
    graph = Graph(store=store)

    config = OntologyConfig()
    config.multimodal_config = MultimodalConfig()
    ontology_manager = OntologyManager(config, graph)
    dataset_manager = DatasetManager(ontology_manager)
    dataset_manager.initialise(glob_path="data/datasets/ALS/**/*.csv")
    topic_man = GuidanceManager(
        ontology_manager, conn_str=setup.conn_str, llm_model_id=setup.model_id
    )
    topic_man.llama_model
    print(topic_man.identifier)

    topic_init = TopicInitator(topic_man)
    topic_init.initate(force=True, delete_tables=True)

    llm_man = LLMQuery(topic_man)
    llm_man.initiate(force=True)

-- Loading and merging datasets
0 datasets loaded
Using device cuda
Loading LLM model NousResearch/Hermes-3-Llama-3.2-3B-GGUF None


llama_model_loader: loaded meta data with 38 key-value pairs and 255 tensors from /home/bkantz/.cache/huggingface/hub/models--NousResearch--Hermes-3-Llama-3.2-3B-GGUF/snapshots/3cd927095d8cbab12c743f932aa63b6f7bbfa141/./Hermes-3-Llama-3.2-3B.Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Hermes 3 Llama 3.2 3b Base Fft Chatml...
llama_model_loader: - kv   3:                            general.version str              = 2024-11-24
llama_model_loader: - kv   4:                       general.organization str              = Nous Research Core
llama_model_loader: - kv   5:                           general.finetune str    

9dcd1ca8dd910b566efdb1f6a76eaacb3c2187b9718c6be6be696212682c6c9b
Initializing topics


Saving anonymous properties: 0it [00:00, ?it/s]
Saving classes:   0%|          | 0/59 [00:00<?, ?it/s]Some weights of the model checkpoint at dunzhang/stella_en_400M_v5 were not used when initializing NewModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Saving classes:  71%|███████   | 42/59 [00:22<00:09,  1.87it/s]


TypeError: 'int' object is not subscriptable

In [10]:
ontology_manager.get_named_individuals("bn74")

[]